# Agent Colab 算力端
顺序跑三个 cell：挂 Drive → 引导（装网+代码+依赖+权重+拉起+契约）→ 保活。
前置：`AgentColab/secrets/colab.env` 有 HF_TOKEN / TS_AUTHKEY / INFER_API_KEY。
首次布置模型用 `stage_to_drive.sh`（CPU 机型即可）；本笔记本要 GPU（A100）。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
MODEL = 'google/gemma-4-31B-it'  #@param ['google/gemma-4-31B-it', 'google/gemma-4-26B-A4B-it', 'Qwen/Qwen3.6-35B-A3B', 'ornith-ai/Ornith-1.0-35B']
!bash /content/drive/MyDrive/AgentColab/bootstrap.sh $MODEL

In [ ]:
# 保活 + 心跳仪表：每 5 分钟打一行（服务健康 / GPU 显存 / tailnet 状态）
import subprocess, time
while True:
    h = subprocess.run(['curl', '-s', '-m', '3', 'http://127.0.0.1:14161/health'],
                       capture_output=True, text=True).stdout.strip() or 'DOWN'
    g = subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.total',
                        '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
    print(time.strftime('%H:%M:%S'), '| health:', h[:60], '| vram:', g)
    time.sleep(300)